# Option A: Goldset Chunk-Level from V3 Chunks

Generates chunk-level questions from **V3 chunks** (`rag_chunks_test`), then maps back to V1.

This is the mirror of `goldset_chunk_level_generation.ipynb` (which generated from V1 chunks).
If V3 wins on its own goldset (as expected due to bias) AND V1 won on its own goldset,
it confirms that chunk-level evaluation is inherently biased toward the source chunking.

**Steps**:
1. Load V3 chunks from Service-Public documents
2. Reconstruct documents with V3 chunk markers
3. Generate chunk-level Q/A with LLM (GPT-4.1-mini)
4. Map each question to best V1 chunk via LLM
5. Save as goldset `chunk_level_v3`

**Filter**: Service-Public only.

In [ ]:
# =============================================================================
# Cell 1 — IMPORTS & SETUP
# =============================================================================
import os, sys, json, time, re
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import psycopg
from psycopg.rows import dict_row
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / '.env')

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# =============================================================================
# Cell 2 — DB, EMBEDDER & LLM
# =============================================================================
from openai import OpenAI
from src.rag.embedder import AlbertEmbedder

DSN = os.getenv("TUNNEL_DSN") or os.getenv("SCALINGO_POSTGRESQL_URL") or os.getenv("PG_DSN")
embedder = AlbertEmbedder(timeout=15, normalize=True)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
LLM_MODEL = "gpt-4.1-mini"

GOLDSET_NAME = "chunk_level_v3"

with psycopg.connect(DSN, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute('SELECT count(*) as cnt FROM goldset_questions_v2')
        q_count = cur.fetchone()['cnt']

print(f"DB: {q_count} questions | LLM: {LLM_MODEL} | Goldset: {GOLDSET_NAME}")

In [ ]:
# =============================================================================
# Cell 3 — LOAD V3 CHUNKS FOR SERVICE-PUBLIC DOCS
# =============================================================================

def questions_for_doc(n_chunks: int) -> int:
    """How many questions to generate per document based on chunk count."""
    if n_chunks <= 5: return 2
    if n_chunks <= 15: return 3
    if n_chunks <= 30: return 4
    if n_chunks <= 60: return 5
    return 6

# Get common_corpus SP short_ids
with psycopg.connect(DSN, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT DISTINCT gq.gold_sources as short_id
            FROM goldset_questions_v2 gq
            JOIN rag_documents d ON gq.gold_sources = d.short_id
            WHERE 'common_corpus' = ANY(gq.tags)
              AND d.publisher = 'Service-Public'
        """)
        sp_short_ids = [r['short_id'] for r in cur.fetchall()]
        print(f"Service-Public common_corpus: {len(sp_short_ids)} documents")

        # Load V3 chunks for each document
        docs = {}
        for sid in sp_short_ids:
            cur.execute("""
                SELECT c.chunk_id::text, c.chunk_markdown, c.chunk_index,
                       c.token_count, s.heading_path, s.section_index,
                       d.title as source_name
                FROM rag_chunks_test c
                JOIN rag_sections s ON c.section_id = s.section_id
                JOIN rag_documents d ON c.doc_id = d.doc_id
                WHERE d.short_id = %s
                ORDER BY s.section_index, c.chunk_index
            """, [sid])
            rows = [dict(r) for r in cur.fetchall()]
            if rows:
                docs[sid] = {
                    'source_name': rows[0]['source_name'],
                    'chunks': rows,
                }

total_q = sum(questions_for_doc(len(d['chunks'])) for d in docs.values())
print(f"\nLoaded {len(docs)} documents:")
for sid, d in sorted(docs.items(), key=lambda x: -len(x[1]['chunks']))[:10]:
    nq = questions_for_doc(len(d['chunks']))
    print(f"  {sid:15s} | {len(d['chunks']):3d} V3 chunks | {nq} questions")
print(f"  ...")
print(f"\nEstimated total questions: {total_q}")

In [ ]:
# =============================================================================
# Cell 4 — DOCUMENT RECONSTRUCTION WITH V3 CHUNK MARKERS
# =============================================================================

MAX_DOC_CHARS = 25000

def reconstruct_v3_document(chunks: list[dict], max_chars: int = MAX_DOC_CHARS) -> str:
    """Reconstruct document from V3 chunks with ID markers."""
    parts = []
    total_chars = 0
    
    for i, chunk in enumerate(chunks):
        separator = f"\n{'='*60}\n[CHUNK {i+1} | ID: {chunk['chunk_id']}]\n{'='*60}\n"
        chunk_text = chunk['chunk_markdown']
        
        if total_chars + len(separator) + len(chunk_text) > max_chars:
            parts.append(f"\n[... document tronqué après {i} chunks sur {len(chunks)} ...]\n")
            break
        
        parts.append(separator)
        parts.append(chunk_text)
        total_chars += len(separator) + len(chunk_text)
    
    return "".join(parts)


# Test
test_sid = list(docs.keys())[0]
test_doc = reconstruct_v3_document(docs[test_sid]['chunks'])
print(f"Test: '{test_sid}' ({len(docs[test_sid]['chunks'])} chunks) → {len(test_doc):,} chars")
print(test_doc[:800])

In [ ]:
# =============================================================================
# Cell 5 — GENERATION PROMPT
# =============================================================================

SYSTEM_PROMPT = """Tu es un expert en création de datasets d'évaluation pour des systèmes RAG dans le domaine des ressources humaines de la fonction publique française.

Ton rôle est de générer des paires question/réponse **au niveau chunk** : chaque question doit être liée à UN chunk spécifique dont elle teste la récupération.

RÈGLES CRITIQUES :

1. **Spécificité au chunk** : Chaque question doit pouvoir être répondue UNIQUEMENT avec les informations contenues dans le chunk indiqué. Un autre chunk du même document ne doit PAS contenir la réponse.

2. **Précision factuelle** : Les questions doivent porter sur des DÉTAILS PRÉCIS et FACTUELS :
   - Montants, seuils, pourcentages spécifiques
   - Délais, durées, dates précises
   - Conditions d'éligibilité détaillées
   - Références à des articles de loi ou décrets
   - Exceptions ou cas particuliers

3. **Diversité des chunks ciblés** : Répartis tes questions sur des chunks DIFFÉRENTS. Ne génère pas 2 questions sur le même chunk.

4. **Réponses auto-suffisantes** : 2-5 phrases, factuelles. JAMAIS de référence implicite au document/chunk.
   - BON : "Selon l'article L.332-22 du CGFP, la durée maximale est de 6 mois."
   - MAUVAIS : "La fiche indique...", "Ce paragraphe précise..."

5. **Questions auto-suffisantes** : Se comprennent SEULES.
   - BON : "Quelle est la durée maximale d'un contrat de projet selon le CGFP ?"
   - MAUVAIS : "Que dit ce document sur...?"

6. **Format strict JSON** : Retourne UNIQUEMENT un tableau JSON."""


def make_user_prompt(source_name: str, reconstructed_doc: str, n_questions: int, chunk_ids: list[str]) -> str:
    chunk_id_list = "\n".join([f"  - {cid}" for cid in chunk_ids[:30]])
    return f"""Voici un document RH reconstitué à partir de ses chunks.
Chaque chunk est délimité par des séparateurs indiquant son ID unique.

**Document** : {source_name}

---
{reconstructed_doc}
---

IDs des chunks disponibles :
{chunk_id_list}

Génère exactement {n_questions} paires question/réponse. Chaque question doit :
- Être liée à UN SEUL chunk spécifique (identifié par son ID)
- Porter sur un détail factuel PRÉCIS contenu UNIQUEMENT dans ce chunk

Retourne un tableau JSON :
```json
[
  {{"question": "...", "answer": "...", "gold_chunk_id": "<uuid exact>"}}
]
```

IMPORTANT : Utilise les IDs EXACTS, chaque question cible un chunk DIFFÉRENT."""


print(f"Prompt ready ({len(SYSTEM_PROMPT)} chars)")

In [ ]:
# =============================================================================
# Cell 6 — TEST ON ONE DOCUMENT
# =============================================================================

test_sid = list(docs.keys())[0]
test_d = docs[test_sid]
n_q = questions_for_doc(len(test_d['chunks']))
doc_text = reconstruct_v3_document(test_d['chunks'])
chunk_ids = [c['chunk_id'] for c in test_d['chunks']]

prompt = make_user_prompt(test_d['source_name'], doc_text, n_q, chunk_ids)
print(f"Doc: {test_sid} | {len(test_d['chunks'])} chunks | {n_q} questions")
print(f"Prompt: {len(prompt):,} chars")

response = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ],
    max_tokens=2000, temperature=0.3,
)

raw = response.choices[0].message.content.strip()
if raw.startswith('```'):
    raw = raw.split('\n', 1)[-1].rsplit('```', 1)[0].strip()

qa_pairs = json.loads(raw)
print(f"\nGenerated {len(qa_pairs)} Q/A pairs:")
for qa in qa_pairs:
    valid = qa['gold_chunk_id'] in chunk_ids
    print(f"  {'✓' if valid else '✗'} chunk={qa['gold_chunk_id'][:16]}...")
    print(f"    Q: {qa['question'][:100]}")
    print(f"    A: {qa['answer'][:100]}")
    print()

In [ ]:
# =============================================================================
# Cell 7 — GENERATE FOR ALL DOCUMENTS
# =============================================================================

all_qa = []  # {short_id, source_name, question, answer, gold_chunk_id}
errors = []
t_start = time.time()

for idx, (sid, d) in enumerate(docs.items()):
    n_q = questions_for_doc(len(d['chunks']))
    doc_text = reconstruct_v3_document(d['chunks'])
    chunk_ids = [c['chunk_id'] for c in d['chunks']]
    prompt = make_user_prompt(d['source_name'], doc_text, n_q, chunk_ids)
    
    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ],
            max_tokens=2000, temperature=0.3,
        )
        raw = response.choices[0].message.content.strip()
        if raw.startswith('```'):
            raw = raw.split('\n', 1)[-1].rsplit('```', 1)[0].strip()
        qa_pairs = json.loads(raw)
    except Exception as e:
        errors.append({'short_id': sid, 'error': str(e)})
        print(f"  [{idx+1}/{len(docs)}] ERROR {sid}: {e}")
        time.sleep(1)
        continue
    
    # Validate chunk_ids
    valid = 0
    for qa in qa_pairs:
        if qa.get('gold_chunk_id') in chunk_ids:
            all_qa.append({
                'short_id': sid,
                'source_name': d['source_name'],
                'question': qa['question'],
                'answer': qa['answer'],
                'gold_chunk_id': qa['gold_chunk_id'],
            })
            valid += 1
        else:
            errors.append({'short_id': sid, 'error': f"Invalid chunk_id: {qa.get('gold_chunk_id', '?')[:20]}"})
    
    print(f"  [{idx+1}/{len(docs)}] {sid} | {len(d['chunks'])} chunks | {valid}/{len(qa_pairs)} valid Q/A")
    time.sleep(0.3)

elapsed = time.time() - t_start
print(f"\n{'='*60}")
print(f"Generated {len(all_qa)} questions in {elapsed:.0f}s")
print(f"Errors: {len(errors)}")

In [ ]:
# =============================================================================
# Cell 8 — SAVE QUESTIONS TO DB
# =============================================================================

print(f"Saving {len(all_qa)} questions as goldset '{GOLDSET_NAME}'...")

saved = 0
with psycopg.connect(DSN, autocommit=False) as conn:
    with conn.cursor() as cur:
        for qa in all_qa:
            comment = json.dumps({
                'gold_chunk_id': qa['gold_chunk_id'],
                'table': 'rag_chunks_test',
                'source': 'v3_generation',
            }, ensure_ascii=False)
            
            cur.execute("""
                INSERT INTO goldset_questions_v2
                    (goldset_name, question, gold_answer, gold_sources, comment)
                VALUES (%s, %s, %s, %s, %s)
            """, [GOLDSET_NAME, qa['question'], qa['answer'], qa['short_id'], comment])
            saved += 1
        conn.commit()

print(f"Saved {saved} questions")

In [ ]:
# =============================================================================
# Cell 9 — EMBED QUESTIONS
# =============================================================================

with psycopg.connect(DSN, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT id, question FROM goldset_questions_v2
            WHERE goldset_name = %s AND embedding_albert IS NULL
            ORDER BY id
        """, [GOLDSET_NAME])
        to_embed = cur.fetchall()

print(f"{len(to_embed)} questions to embed")

embedded = 0
with psycopg.connect(DSN, autocommit=False) as conn:
    with conn.cursor() as cur:
        for i, q in enumerate(to_embed):
            vec = embedder.embed_query(q['question'])
            cur.execute(
                "UPDATE goldset_questions_v2 SET embedding_albert = %s WHERE id = %s",
                [vec, q['id']]
            )
            embedded += 1
            if (i + 1) % 20 == 0:
                print(f"  Embedded {i+1}/{len(to_embed)}")
                time.sleep(0.5)
        conn.commit()

print(f"Embedded {embedded} questions")

In [ ]:
# =============================================================================
# Cell 10 — MAP TO V1 CHUNKS VIA LLM (vector pre-filter + LLM)
# =============================================================================

MAPPING_PROMPT = """Tu es un expert en analyse de documents. On te donne une question, un chunk V3 (gold), et 5 chunks V1 candidats. Identifie le meilleur match V1.

RÈGLES:
1. Compare le CONTENU du chunk V3 avec chaque chunk V1.
2. Choisis le V1 qui couvre les mêmes informations.
3. Si AUCUN ne correspond, réponds null.

RÉPONDS en JSON:
{"chunk_id": "<hash_id>", "confidence": "high|medium|low", "reason": "<30 mots>"}
ou {"chunk_id": null, "confidence": "none", "reason": "<30 mots>"}"""


with psycopg.connect(DSN, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT id, question, gold_sources, comment, embedding_albert
            FROM goldset_questions_v2
            WHERE goldset_name = %s AND embedding_albert IS NOT NULL
            ORDER BY id
        """, [GOLDSET_NAME])
        v3_questions = cur.fetchall()

print(f"{len(v3_questions)} questions to map to V1")

v1_mappings = []
v3_text_cache = {}  # chunk_id -> text
t_start = time.time()

with psycopg.connect(DSN, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        for i, q in enumerate(v3_questions):
            sid = q['gold_sources']
            meta = json.loads(q['comment']) if isinstance(q['comment'], str) else {}
            gold_v3_id = meta.get('gold_chunk_id')
            
            # Load V3 gold chunk text (for LLM context)
            if gold_v3_id and gold_v3_id not in v3_text_cache:
                cur.execute("SELECT chunk_markdown FROM rag_chunks_test WHERE chunk_id = %s::uuid", [gold_v3_id])
                row = cur.fetchone()
                v3_text_cache[gold_v3_id] = row['chunk_markdown'] if row else ''
            v3_text = v3_text_cache.get(gold_v3_id, '')
            
            # Embed V3 gold chunk for vector pre-filter
            if v3_text:
                v3_vec = embedder.embed_query(v3_text)
                time.sleep(0.1)
            else:
                v3_vec = q['embedding_albert']  # fallback to question embedding
            
            # Vector search V1 candidates
            cur.execute("""
                SELECT hash_id as chunk_id, text as chunk_text, section_path,
                       1 - (embedding_m3 <=> %s::vector) as score
                FROM rag_chunks_service_public
                WHERE short_id = %s AND embedding_m3 IS NOT NULL
                ORDER BY embedding_m3 <=> %s::vector
                LIMIT 5
            """, [v3_vec, sid, v3_vec])
            v1_cands = [dict(r) for r in cur.fetchall()]
            
            if not v1_cands:
                v1_mappings.append({'question_id': q['id'], 'v1_chunk_id': None, 'confidence': 'none'})
                print(f"  [{i+1}/{len(v3_questions)}] No V1 chunks for {sid}")
                continue
            
            # LLM pick
            parts = [f"QUESTION: {q['question']}\n"]
            parts.append(f"CHUNK V3 (gold):\n{v3_text[:500]}\n")
            parts.append(f"--- 5 CANDIDATS V1 ---\n")
            for j, c in enumerate(v1_cands):
                parts.append(f"[{j+1}] ID: {c['chunk_id']} | score: {c['score']:.4f}\n{c['chunk_text'][:400]}\n")
            
            try:
                resp = client.chat.completions.create(
                    model=LLM_MODEL,
                    messages=[{"role": "system", "content": MAPPING_PROMPT}, {"role": "user", "content": "\n".join(parts)}],
                    max_tokens=200, temperature=0.0,
                )
                raw = resp.choices[0].message.content.strip()
                if raw.startswith('```'):
                    raw = raw.split('\n', 1)[-1].rsplit('```', 1)[0].strip()
                result = json.loads(raw)
            except Exception as e:
                result = {'chunk_id': None, 'confidence': 'error', 'reason': str(e)[:50]}
            
            v1_id = result.get('chunk_id')
            if v1_id and v1_id not in [c['chunk_id'] for c in v1_cands]:
                v1_id = None
            
            v1_mappings.append({
                'question_id': q['id'],
                'v1_chunk_id': v1_id,
                'confidence': result.get('confidence', 'error'),
                'reason': result.get('reason', ''),
            })
            
            status = v1_id[:12] + '...' if v1_id else 'NO_MATCH'
            print(f"  [{i+1}/{len(v3_questions)}] {sid} -> V1={status} ({result.get('confidence','?')})")
            time.sleep(0.2)

elapsed = time.time() - t_start
n_mapped = sum(1 for m in v1_mappings if m['v1_chunk_id'])
print(f"\nMapped {n_mapped}/{len(v1_mappings)} to V1 in {elapsed:.0f}s")

In [ ]:
# =============================================================================
# Cell 11 — SAVE V1 MAPPINGS TO DB
# =============================================================================

saved = 0
with psycopg.connect(DSN, autocommit=False, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        for m in v1_mappings:
            cur.execute("SELECT comment FROM goldset_questions_v2 WHERE id = %s", [m['question_id']])
            row = cur.fetchone()
            try:
                comment_data = json.loads(row['comment']) if row and row['comment'] else {}
            except:
                comment_data = {}
            
            # For V3-generated goldset:
            # gold_chunk_id = the V3 chunk (source) → used by V3 strategies as gold_chunk_id_v3
            # We store V1 mapping separately so the eval page can use it
            # Convention: gold_chunk_id_v3 = V3 gold, gold_chunk_id = V1 gold (for DE strategies)
            comment_data['gold_chunk_id_v3'] = comment_data.get('gold_chunk_id')  # V3 gold (source)
            comment_data['gold_chunk_id'] = m['v1_chunk_id']  # V1 gold (mapped)
            comment_data['v1_confidence'] = m['confidence']
            comment_data['v1_reason'] = m.get('reason', '')
            comment_data['v3_confidence'] = 'high'  # V3 is the source, always high
            comment_data['v3_method'] = 'source'
            
            cur.execute(
                "UPDATE goldset_questions_v2 SET comment = %s WHERE id = %s",
                [json.dumps(comment_data, ensure_ascii=False), m['question_id']]
            )
            saved += 1
        conn.commit()

print(f"Saved {saved} V1 mappings")

In [ ]:
# =============================================================================
# Cell 12 — VERIFICATION
# =============================================================================

with psycopg.connect(DSN, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT id, question, gold_sources, comment
            FROM goldset_questions_v2
            WHERE goldset_name = %s
            ORDER BY id
        """, [GOLDSET_NAME])
        verified = cur.fetchall()

n_total = len(verified)
n_v1 = 0
n_v3 = 0
n_both = 0

for q in verified:
    try:
        meta = json.loads(q['comment']) if isinstance(q['comment'], str) else {}
    except:
        meta = {}
    has_v1 = bool(meta.get('gold_chunk_id'))
    has_v3 = bool(meta.get('gold_chunk_id_v3'))
    n_v1 += has_v1
    n_v3 += has_v3
    n_both += (has_v1 and has_v3)

print(f"Goldset '{GOLDSET_NAME}': {n_total} questions")
print(f"  V3 gold chunk (source): {n_v3}")
print(f"  V1 gold chunk (mapped): {n_v1}")
print(f"  Both: {n_both}")
print(f"\nReady for evaluation!")